# NTR Arogyaseva

## Auditoría inicial

In [5]:
from notebook_utils import ensure_repo_root

# Establish the repository root as the working directory for this notebook
ensure_repo_root()

WindowsPath('C:/Github/NTR-Arogyaseva-analysis')

In [6]:
from src.utils.io import load_data, set_pandas_display_options

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Establecer opciones de visualización para pandas
set_pandas_display_options()

# Importar el dataset
df = load_data(stage="raw")

In [7]:
print(f"Dataset shape: {df.shape}")
print("\n First rows")
display(df.head())
print("\n Columns information")
display(df.info())
print("\n Numerical variables description")
display(df.describe())

Dataset shape: (479688, 24)

 First rows


,,AGE,SEX,CASTE_NAME,CATEGORY_CODE,CATEGORY_NAME,SURGERY_CODE,SURGERY,VILLAGE,MANDAL_NAME,DISTRICT_NAME,PREAUTH_DATE,PREAUTH_AMT,CLAIM_DATE,CLAIM_AMOUNT,HOSP_NAME,HOSP_TYPE,HOSP_LOCATION,HOSP_DISTRICT,SURGERY_DATE,DISCHARGE_DATE,Mortality Y / N,MORTALITY_DATE,SRC_REGISTRATION
0,1,56,Female,BC,M6,NEPHROLOGY,M6.5,Maintenance Hemodialysis For Crf,Lolugu,Ponduru,Srikakulam,03/08/2013 20:38:48,12500,22/03/2017 20:25:18,11000,"Rims Govt. General Hospital, Srikakulam",G,SRIKAKULAM,Srikakulam,06/08/2013 00:00:00,07/09/2013 00:00:00,NO,NaN,D
1,2,37,Male,BC,M6,NEPHROLOGY,M6.5,Maintenance Hemodialysis For Crf,Borivanka,Kaviti,Srikakulam,06/08/2013 07:26:15,12500,22/03/2017 20:25:18,11000,"Rims Govt. General Hospital, Srikakulam",G,SRIKAKULAM,Srikakulam,08/08/2013 00:00:00,09/09/2013 00:00:00,NO,NaN,D
2,3,50,Male,BC,M6,NEPHROLOGY,M6.5,Maintenance Hemodialysis For Crf,Kapasakuddi,Kaviti,Srikakulam,09/08/2013 18:30:50,12500,22/03/2017 20:25:18,11500,"Rims Govt. General Hospital, Srikakulam",G,SRIKAKULAM,Srikakulam,15/08/2013 00:00:00,18/10/2013 00:00:00,NO,NaN,D
3,4,45,Male,BC,M6,NEPHROLOGY,M6.5,Maintenance Hemodialysis For Crf,Telikipenta,Sarubujjili,Srikakulam,24/08/2013 19:37:41,12500,22/03/2017 20:25:18,11000,"Rims Govt. General Hospital, Srikakulam",G,SRIKAKULAM,Srikakulam,24/08/2013 00:00:00,27/09/2013 00:00:00,NO,NaN,D
4,5,54,Male,BC,M6,NEPHROLOGY,M6.5,Maintenance Hemodialysis For Crf,Thandemvalasa,Srikakulam,Srikakulam,28/08/2013 17:03:07,12500,22/03/2017 20:25:19,11000,"Rims Govt. General Hospital, Srikakulam",G,SRIKAKULAM,Srikakulam,31/08/2013 00:00:00,02/10/2013 00:00:00,NO,NaN,D



 Columns information
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 479688 entries, 0 to 479687
Data columns (total 24 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0                     479688 non-null  int64 
 1   AGE               479688 non-null  int64 
 2   SEX               479688 non-null  object
 3   CASTE_NAME        479688 non-null  object
 4   CATEGORY_CODE     479688 non-null  object
 5   CATEGORY_NAME     479688 non-null  object
 6   SURGERY_CODE      479688 non-null  object
 7   SURGERY           479688 non-null  object
 8   VILLAGE           479688 non-null  object
 9   MANDAL_NAME       479688 non-null  object
 10  DISTRICT_NAME     479688 non-null  object
 11  PREAUTH_DATE      479688 non-null  object
 12  PREAUTH_AMT       479688 non-null  int64 
 13  CLAIM_DATE        479688 non-null  object
 14  CLAIM_AMOUNT      479688 non-null  int64 
 15  HOSP_NAME         479688 non-null  object
 16  HOSP_TYPE       

None


 Numerical variables description


,,AGE,PREAUTH_AMT,CLAIM_AMOUNT
count,479688.00,479688.00,479688.00,479688.00
mean,239844.50,44.91,30428.94,27652.21
std,138474.14,18.96,27441.59,25951.71
min,1.00,0.00,5.00,2.00
25%,119922.75,34.00,12500.00,12500.00
50%,239844.50,47.00,25000.00,23200.00
75%,359766.25,59.00,35000.00,30600.00
max,479688.00,107.00,520000.00,520000.00


claim amount podria distribuirse Poisson(lambda) ?? capaz podemos aplicar poisson regression

la primera columna "" parecen ser los IDs. no veo, de primeras, ninguna clave natural candidata

analizar cardinalidad de category_name:category_code, surgery:surgery_code (para ver si contienen la misma información)

qué son los amounts? cual es la relación teórica entre ellos

hay que pasar a date/datetime todos los que terminan en "_DATE"

In [8]:
def profile_missingness(
    df: pd.DataFrame,
    warn_threshold: float = 0.05,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Profile missing values across dataframe columns.
    """

    missingness = pd.DataFrame({
        "column": df.columns,
        "null_count": [df[col].isna().sum() for col in df.columns],
        "null_pct": [round(df[col].isna().mean() * 100, 2) for col in df.columns],
    })

    missingness["status"] = missingness["null_pct"].apply(
        lambda x:
            "critical" if x >= 50
            else "warning" if x >= warn_threshold * 100
            else "ok"
    )

    missingness = (
        missingness
        .sort_values("null_pct", ascending=False)
        .reset_index(drop=True)
    )

    critical_cols = (missingness["status"] == "critical").sum()
    warning_cols = (missingness["status"] == "warning").sum()
    if verbose:
        print(
            f"{critical_cols} critical and "
            f"{warning_cols} warning-level columns detected."
        )

    return missingness

In [9]:
profile_missingness(df)

1 critical and 0 warning-level columns detected.


,column,null_count,null_pct,status
0,MORTALITY_DATE,469566,97.89,critical
1,DISCHARGE_DATE,4560,0.95,ok
2,,0,0.00,ok
3,AGE,0,0.00,ok
4,CATEGORY_CODE,0,0.00,ok
5,CATEGORY_NAME,0,0.00,ok
6,SEX,0,0.00,ok
7,CASTE_NAME,0,0.00,ok
8,SURGERY,0,0.00,ok
9,SURGERY_CODE,0,0.00,ok


In [10]:
def profile_categoricals(
    df: pd.DataFrame,
    columns: list[str],
    top_n: int = 10,
) -> dict:
    """
    Profile categorical distributions and cardinality.
    """

    profiles = {}

    for col in columns:

        value_counts = (
            df[col]
            .value_counts(dropna=False)
            .rename_axis(col)
            .reset_index(name="count")
        )

        value_counts["pct"] = (
            value_counts["count"] / len(df) * 100
        )

        profiles[col] = {
            "n_unique": df[col].nunique(dropna=True),
            "top_categories": value_counts.head(top_n),
        }

        print(
            f"`{col}` → "
            f"{df[col].nunique(dropna=True)} unique categories detected."
        )

    return profiles

In [11]:
categorical_profiles = profile_categoricals(df, df.select_dtypes(include="object").columns.tolist())

`SEX` → 6 unique categories detected.
`CASTE_NAME` → 6 unique categories detected.
`CATEGORY_CODE` → 29 unique categories detected.
`CATEGORY_NAME` → 29 unique categories detected.
`SURGERY_CODE` → 925 unique categories detected.
`SURGERY` → 923 unique categories detected.
`VILLAGE` → 11801 unique categories detected.
`MANDAL_NAME` → 711 unique categories detected.
`DISTRICT_NAME` → 13 unique categories detected.
`PREAUTH_DATE` → 451607 unique categories detected.
`CLAIM_DATE` → 111513 unique categories detected.
`HOSP_NAME` → 467 unique categories detected.
`HOSP_TYPE` → 2 unique categories detected.
`HOSP_LOCATION` → 61 unique categories detected.
`HOSP_DISTRICT` → 20 unique categories detected.
`SURGERY_DATE` → 937 unique categories detected.
`DISCHARGE_DATE` → 918 unique categories detected.
`Mortality Y / N` → 2 unique categories detected.
`MORTALITY_DATE` → 740 unique categories detected.
`SRC_REGISTRATION` → 4 unique categories detected.


In [12]:
for col, profile in categorical_profiles.items():
    if not(col.endswith("DATE")):
        print(f"Column: {col}. Number of unique categories: {profile['n_unique']}.")
        display(profile["top_categories"])

Column: SEX. Number of unique categories: 6.


,SEX,count,pct
0,Male,260718,54.35
1,Female,178947,37.30
2,Male(Child),25068,5.23
3,Female(Child),14925,3.11
4,FEMALE,21,0.00
5,MALE,9,0.00


Column: CASTE_NAME. Number of unique categories: 6.


,CASTE_NAME,count,pct
0,BC,246164,51.32
1,OC,114123,23.79
2,SC,76742,16.00
3,Minorities,29150,6.08
4,ST,13138,2.74
5,Others,371,0.08


Column: CATEGORY_CODE. Number of unique categories: 29.


...

,HOSP_DISTRICT,count,pct
0,Guntur,60988,12.71
1,Vishakhapatnam,59334,12.37
2,East Godavari,58065,12.10
3,Chittoor,49172,10.25
4,Krishna,45174,9.42
5,Nellore,42405,8.84
6,Kurnool,33024,6.88
7,Hyderabad,27915,5.82
8,West Godavari,23769,4.96
9,Anantapur,18444,3.84


Column: Mortality Y / N. Number of unique categories: 2.


,Mortality Y / N,count,pct
0,NO,469566,97.89
1,YES,10122,2.11


Column: SRC_REGISTRATION. Number of unique categories: 4.


,SRC_REGISTRATION,count,pct
0,D,437948,91.30
1,P,19100,3.98
2,CMO,14706,3.07
3,MC,7934,1.65


relaciones 1:N
* category - surgery
* district - location

relaciones 1:1

* category_code - category_name, 
* surgery - surgery_code

In [ ]:
(df["HOSP_DISTRICT"] == df["DISTRICT_NAME"]).mean()

np.float64(0.7148917629792698)

district_name debe referirse al distrito del paciente

In [ ]:
def check_one_to_one_mapping(
    df: pd.DataFrame,
    key_col: str,
    value_col: str,
) -> pd.DataFrame:
    """
    Validate one-to-one consistency between two columns.
    """

    mapping_check = (
        df.groupby(key_col)[value_col]
        .nunique(dropna=True)
        .reset_index(name="unique_values")
    )

    inconsistent = (
        mapping_check
        .query("unique_values > 1")
        .sort_values("unique_values", ascending=False)
        .reset_index(drop=True)
    )

    if inconsistent.empty:
        print(
            f"No inconsistencies detected between "
            f"`{key_col}` and `{value_col}`."
        )

        return pd.DataFrame({})
    else:
        print(
            f"{len(inconsistent)} `{key_col}` values are associated "
            f"with multiple `{value_col}` values."
        )

        return inconsistent

### RELACIÓN district_name - village. 1:N, 1:1

In [ ]:
check_one_to_one_mapping(df, "DISTRICT_NAME", "VILLAGE")

13 `DISTRICT_NAME` values are associated with multiple `VILLAGE` values.


,DISTRICT_NAME,unique_values
0,Chittoor,1404
1,Srikakulam,1392
2,Vishakhapatnam,1336
3,East Godavari,1132
4,Vizianagaram,1124
5,Nellore,1067
6,Anantapur,1000
7,Krishna,930
8,Prakasam,911
9,YSR Kadapa,871


In [ ]:
check_one_to_one_mapping(df, "VILLAGE", "DISTRICT_NAME")

986 `VILLAGE` values are associated with multiple `DISTRICT_NAME` values.


,VILLAGE,unique_values
0,Kothapalle,11
1,Krishnapuram,10
2,Gangavaram,10
3,Ward-1,9
4,Venkatapuram,9
...,...,...
981,Jupudi,2
982,Julakallu,2
983,Jonnalagadda,2
984,Yerrampeta,2


village no pertenece a un district_name único

### relación category - surgery. 1:N, 1:1

In [ ]:
check_one_to_one_mapping(df, "CATEGORY_NAME", "SURGERY")

26 `CATEGORY_NAME` values are associated with multiple `SURGERY` values.


,CATEGORY_NAME,unique_values
0,SURGICAL ONCOLOGY,113
1,CARDIAC AND CARDIOTHORACIC SURGERY,102
2,GENERAL SURGERY,85
3,MEDICAL ONCOLOGY,78
4,PEDIATRICS,68
5,NEUROSURGERY,61
6,GENITO URINARY SURGERIES,59
7,PEDIATRIC SURGERIES,55
8,SURGICAL GASTRO ENTEROLOGY,52
9,PLASTIC SURGERY,31


In [ ]:
check_one_to_one_mapping(df, "SURGERY", "CATEGORY_NAME")

2 `SURGERY` values are associated with multiple `CATEGORY_NAME` values.


,SURGERY,unique_values
0,Cholecystectomy,2
1,Decompression/Excision Of Optic Nerve Lesions,2


parece haber inconsistencias en la relación surgery - category 1:1. pero son poquitas

### relación category_name - category_code 1:1, 1:1

In [ ]:
check_one_to_one_mapping(df, "CATEGORY_CODE", "CATEGORY_NAME")

No inconsistencies detected between `CATEGORY_CODE` and `CATEGORY_NAME`.


""


In [ ]:
check_one_to_one_mapping(df, "CATEGORY_NAME", "CATEGORY_CODE")

No inconsistencies detected between `CATEGORY_NAME` and `CATEGORY_CODE`.


""


no parece haber inconsistencias

### relación surgery - surgery_code. 1:1, 1:1

In [ ]:
check_one_to_one_mapping(df, "SURGERY", "SURGERY_CODE")

2 `SURGERY` values are associated with multiple `SURGERY_CODE` values.


,SURGERY,unique_values
0,Cholecystectomy,2
1,Decompression/Excision Of Optic Nerve Lesions,2


In [ ]:
check_one_to_one_mapping(df, "SURGERY_CODE", "SURGERY")

No inconsistencies detected between `SURGERY_CODE` and `SURGERY`.


""


In [ ]:
# Caso 1, misma cirugía, distinta categoría, pero mismo tipo de hospital
display(df[df["SURGERY"]=="Cholecystectomy"]["CATEGORY_NAME"].value_counts())
df[df["SURGERY"]=="Cholecystectomy"]["HOSP_TYPE"].value_counts()

CATEGORY_NAME
GENERAL SURGERY               93
SURGICAL GASTRO ENTEROLOGY     5
Name: count, dtype: int64

HOSP_TYPE
G    98
Name: count, dtype: int64

reemplazar SURGICAL GASTRO ENTEROLOGY por GENERAL SURGERY, debido a que la cirugía se puede categorizar como tal, y en ningún caso visto se encuentra en hospitales especializados como para que entre en una categoría más específica

In [ ]:
# Caso 2, misma cirugía, distinta categoría, pero diferente tipo de hospital
display(df[df["SURGERY"]=="Decompression/Excision Of Optic Nerve Lesions"][["SURGERY", "CATEGORY_NAME", "HOSP_TYPE"]])

,SURGERY,CATEGORY_NAME,HOSP_TYPE
11158,Decompression/Excision Of Optic Nerve Lesions,NEUROSURGERY,C
41789,Decompression/Excision Of Optic Nerve Lesions,NEUROSURGERY,G
218999,Decompression/Excision Of Optic Nerve Lesions,OPHTHALMOLOGY SURGERY,G


no hay razones aparentes para normalizar ninguna de las dos categorías, debería quedar como está.

### relaciones 1:1. hosp_name (identificador del hospital, FK)

In [ ]:
check_one_to_one_mapping(df, "HOSP_NAME", "HOSP_LOCATION")

1 `HOSP_NAME` values are associated with multiple `HOSP_LOCATION` values.


,HOSP_NAME,unique_values
0,SREE HOSPITAL,2


In [ ]:
check_one_to_one_mapping(df, "HOSP_NAME", "HOSP_DISTRICT")

1 `HOSP_NAME` values are associated with multiple `HOSP_DISTRICT` values.


,HOSP_NAME,unique_values
0,SREE HOSPITAL,2


In [ ]:
df[df["HOSP_NAME"]=="SREE HOSPITAL"]["HOSP_DISTRICT"].value_counts()

HOSP_DISTRICT
YSR Kadapa    99
Chittoor      28
Name: count, dtype: int64

In [ ]:
df[df["HOSP_NAME"]=="SREE HOSPITAL"]["HOSP_LOCATION"].value_counts()

HOSP_LOCATION
KADAPA       99
TIRUPATHI    28
Name: count, dtype: int64

In [ ]:
df[df["HOSP_NAME"]=="SREE HOSPITAL"]["HOSP_TYPE"].value_counts()

HOSP_TYPE
C    127
Name: count, dtype: int64

esta inconsistencia es más grave, un mismo hospital (en teoría) tiene registradas dos ubicaciones distintas.

In [ ]:
check_one_to_one_mapping(df, "HOSP_NAME", "HOSP_TYPE")

No inconsistencies detected between `HOSP_NAME` and `HOSP_TYPE`.


""


hospitales están relacionados consistentemente a un solo hosp_type

### relación hosp_district - hosp_location 1:N, 1:1

In [ ]:
check_one_to_one_mapping(df, "HOSP_DISTRICT", "HOSP_LOCATION")

15 `HOSP_DISTRICT` values are associated with multiple `HOSP_LOCATION` values.


,HOSP_DISTRICT,unique_values
0,East Godavari,8
1,West Godavari,8
2,Chittoor,6
3,Prakasam,5
4,Hyderabad,4
5,Krishna,4
6,Guntur,4
7,Vishakhapatnam,3
8,Kurnool,3
9,Ranga Reddy,3


In [ ]:
check_one_to_one_mapping(df, "HOSP_LOCATION", "HOSP_DISTRICT")

2 `HOSP_LOCATION` values are associated with multiple `HOSP_DISTRICT` values.


,HOSP_LOCATION,unique_values
0,HYDERABAD,2
1,SECUNDERABAD,2


In [ ]:
df[df["HOSP_LOCATION"]=="HYDERABAD"]["HOSP_DISTRICT"].value_counts()

HOSP_DISTRICT
Hyderabad      24524
Ranga Reddy     1199
Name: count, dtype: int64

por lo que vi en maps, ranga reddy se superpone en partes con hyderabad, que ese nombre está ligado a la vez (al parecer) a la localidad y a la ciudad (o distrito). no sabría cómo tratar esta inconsistencia

In [ ]:
df[df["HOSP_LOCATION"]=="SECUNDERABAD"]["HOSP_DISTRICT"].value_counts()

HOSP_DISTRICT
Hyderabad      2412
Ranga Reddy       8
Name: count, dtype: int64

siguiendo, secunderabad parece estar por fuera del rango de ranga reddy, eso parece ser directamente erróneo

a pesar de esto, debido a que la ubicación es unicamente información descriptiva de cada hospital, nos interesa principalmente la consistencia entre un ID de hosp (la única candidata es hosp_name) y sus descriptores (tipo, locación, etc.). por lo que no profundizaría en esta cuestión.

### columnas sex y age

In [25]:
df["SEX"].value_counts()

SEX
Male             260718
Female           178947
Male(Child)       25068
Female(Child)     14925
FEMALE               21
MALE                  9
Name: count, dtype: int64

In [26]:
childs = df[df["AGE"]<=14]
childs["SEX"].value_counts()

SEX
Male(Child)      25068
Female(Child)    14925
Male                 1
Name: count, dtype: int64

hay un menor de 14 años que es el único que está cargado como adulto

### full row duplicates

In [ ]:
def check_duplicates(
    df: pd.DataFrame,
    subset: list[str],
) -> pd.DataFrame:
    """
    Return duplicated rows based on subset columns.
    """

    mask = df.duplicated(subset=subset, keep=False)

    if not mask.any():
        print(f"No duplicates found for subset: {', '.join(subset)}") 

        return pd.DataFrame({})
    
    else: 
        print(f"Found {mask.sum()} duplicate rows for subset: {', '.join(subset)}")

        return df.loc[mask].sort_values(subset)

In [ ]:
check_duplicates(df, df.columns.tolist()[1:])

No duplicates found for subset: AGE, SEX, CASTE_NAME, CATEGORY_CODE, CATEGORY_NAME, SURGERY_CODE, SURGERY, VILLAGE, MANDAL_NAME, DISTRICT_NAME, PREAUTH_DATE, PREAUTH_AMT, CLAIM_DATE, CLAIM_AMOUNT, HOSP_NAME, HOSP_TYPE, HOSP_LOCATION, HOSP_DISTRICT, SURGERY_DATE, DISCHARGE_DATE, Mortality Y / N, MORTALITY_DATE, SRC_REGISTRATION


""


## Etapa 2: Limpieza provisional

### index primera columna

In [ ]:
data_clean = df.set_index(df.columns[0], drop=True)

### columnas _DATE a tipo datetime

In [ ]:
def convert_dates(
    df: pd.DataFrame,
    date_cols: list[str],
) -> pd.DataFrame:
    """
    Convert specified columns to datetime format.
    """
    df = df.copy()

    for col in date_cols:
        try:
            df[col] = pd.to_datetime(df[col], format="%d/%m/%Y %H:%M:%S", dayfirst=True, errors="coerce")
            print(f"Column `{col}` successfully converted to datetime.")
        except Exception as e:
            print(f"Error converting column `{col}`: {e}")

    return df

In [ ]:
date_cols = data_clean.select_dtypes(include="object").columns
date_cols = date_cols[date_cols.str.endswith("DATE")]
date_cols

Index(['PREAUTH_DATE', 'CLAIM_DATE', 'SURGERY_DATE', 'DISCHARGE_DATE', 'MORTALITY_DATE'], dtype='object')

In [ ]:
data_clean = convert_dates(data_clean, date_cols)

Column `PREAUTH_DATE` successfully converted to datetime.
Column `CLAIM_DATE` successfully converted to datetime.
Column `SURGERY_DATE` successfully converted to datetime.
Column `DISCHARGE_DATE` successfully converted to datetime.
Column `MORTALITY_DATE` successfully converted to datetime.


verificando de que la conversión no haya generado nulos

In [ ]:
profile_missingness(data_clean)

1 critical and 0 warning-level columns detected.


,column,null_count,null_pct,status
0,MORTALITY_DATE,469566,97.89,critical
1,DISCHARGE_DATE,4562,0.95,ok
2,AGE,0,0.00,ok
3,CATEGORY_CODE,0,0.00,ok
4,CATEGORY_NAME,0,0.00,ok
5,SEX,0,0.00,ok
6,CASTE_NAME,0,0.00,ok
7,SURGERY,0,0.00,ok
8,SURGERY_CODE,0,0.00,ok
9,VILLAGE,0,0.00,ok


### normalización de los nombres de las columnas

In [ ]:
columns = data_clean.columns.str.lower().str.replace(" y / n", "").to_list()
columns

['age',
 'sex',
 'caste_name',
 'category_code',
 'category_name',
 'surgery_code',
 'surgery',
 'village',
 'mandal_name',
 'district_name',
 'preauth_date',
 'preauth_amt',
 'claim_date',
 'claim_amount',
 'hosp_name',
 'hosp_type',
 'hosp_location',
 'hosp_district',
 'surgery_date',
 'discharge_date',
 'mortality',
 'mortality_date',
 'src_registration']

In [ ]:
data_clean.columns = columns
data_clean.head(0)

,age,sex,caste_name,category_code,category_name,surgery_code,surgery,village,mandal_name,district_name,preauth_date,preauth_amt,claim_date,claim_amount,hosp_name,hosp_type,hosp_location,hosp_district,surgery_date,discharge_date,mortality,mortality_date,src_registration
,,,,,,,,,,,,,,,,,,,,,,,


### normalización de columnas de texto (remove accent y lower casing)

In [ ]:
import re
import unicodedata


def remove_accents(text: str) -> str:
    normalized = unicodedata.normalize("NFKD", text)
    return "".join(char for char in normalized if not unicodedata.combining(char))


def normalize_text(text: str) -> str:
    if text is None:
        return text

    text = str(text).strip().lower()
    text = remove_accents(text)
    text = re.sub(r"\s+", " ", text)

    return text


In [ ]:
def normalize_text_columns(
        df: pd.DataFrame, 
        text_cols: list | str,
        normalize_text = normalize_text,
        verbose: bool = False,
    ) -> pd.DataFrame:
    """Normalize text columns in the dataframe using the provided normalization function."""

    df = df.copy()

    if isinstance(text_cols, str):
        text_cols = [text_cols]

    df[text_cols] = df[text_cols].map(normalize_text)

    if verbose:
        print(f"Text columns normalized: {text_cols}")    

    return df

In [ ]:
text_cols = data_clean.select_dtypes(include="object").columns.tolist()
text_cols

['sex',
 'caste_name',
 'category_code',
 'category_name',
 'surgery_code',
 'surgery',
 'village',
 'mandal_name',
 'district_name',
 'hosp_name',
 'hosp_type',
 'hosp_location',
 'hosp_district',
 'mortality',
 'src_registration']

antes

In [ ]:
categorical_profiles = profile_categoricals(data_clean, text_cols)

`sex` → 6 unique categories detected.
`caste_name` → 6 unique categories detected.
`category_code` → 29 unique categories detected.
`category_name` → 29 unique categories detected.
`surgery_code` → 925 unique categories detected.
`surgery` → 923 unique categories detected.
`village` → 11801 unique categories detected.
`mandal_name` → 711 unique categories detected.
`district_name` → 13 unique categories detected.
`hosp_name` → 467 unique categories detected.
`hosp_type` → 2 unique categories detected.
`hosp_location` → 61 unique categories detected.
`hosp_district` → 20 unique categories detected.
`mortality` → 2 unique categories detected.
`src_registration` → 4 unique categories detected.


In [ ]:
data_clean = normalize_text_columns(data_clean, text_cols, verbose=True, normalize_text=normalize_text)

Text columns normalized: ['sex', 'caste_name', 'category_code', 'category_name', 'surgery_code', 'surgery', 'village', 'mandal_name', 'district_name', 'hosp_name', 'hosp_type', 'hosp_location', 'hosp_district', 'mortality', 'src_registration']


después

In [ ]:
categorical_profiles = profile_categoricals(data_clean, text_cols)

`sex` → 4 unique categories detected.
`caste_name` → 6 unique categories detected.
`category_code` → 29 unique categories detected.
`category_name` → 29 unique categories detected.
`surgery_code` → 925 unique categories detected.
`surgery` → 919 unique categories detected.
`village` → 11743 unique categories detected.
`mandal_name` → 711 unique categories detected.
`district_name` → 13 unique categories detected.
`hosp_name` → 465 unique categories detected.
`hosp_type` → 2 unique categories detected.
`hosp_location` → 48 unique categories detected.
`hosp_district` → 20 unique categories detected.
`mortality` → 2 unique categories detected.
`src_registration` → 4 unique categories detected.


## ideas

* regresión multinivel
* análisis de moderación de variables
* targets: amounts y mortality
* category-level, hospital-level, hosp_type-level:

para cada categoría o grupo, analizar la distribución de las variables objetivo, de encontrar diferencias interesantes, profundizar con test de hipótesis multi-clase o con dos clases específicas
* correlación 
* analizar por clases de hosp_type